In [0]:
# =============================================================================
# AI Data Model Designer — One-Click Installer
# =============================================================================
# Just run this cell. It will:
#   1. Auto-discover your SQL Warehouse
#   2. Auto-discover ALL Claude LLM endpoints (dynamic, no hardcoded names)
#   3. Create & deploy the Databricks App
#   4. Print the live app URL
# =============================================================================

import os, time, re, requests
from datetime import datetime
from databricks.sdk import WorkspaceClient

# ---------------------------------------------------------------------------
# Check for optional overrides from Cell 2 (if user set them)
# ---------------------------------------------------------------------------
_override_warehouse = globals().get("WAREHOUSE_ID", None)
_override_endpoints = globals().get("LLM_ENDPOINTS", None)
_override_app_name = globals().get("APP_NAME", None)

APP_NAME = _override_app_name or "ai-data-model-designer"
APP_TITLE = "AI Data Model Designer"
# Subfolder containing app.py, app.yaml, requirements.txt
APP_SUBFOLDER = "app"

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def log(emoji, msg):
    ts = datetime.now().strftime("[%H:%M:%S]")
    print(f"{ts} {emoji} {msg}", flush=True)

def banner(text):
    print("\n" + "\u2501" * 50, flush=True)
    print(f"  {text}", flush=True)
    print("\u2501" * 50 + "\n", flush=True)

# ---------------------------------------------------------------------------
# START
# ---------------------------------------------------------------------------
banner(f"\U0001f680 {APP_TITLE} \u2014 Installer")
start_time = time.time()

try:
    w = WorkspaceClient()
    log("\u2705", "Databricks SDK initialized")
except Exception as e:
    log("\u274c", f"Failed to initialize SDK: {e}")
    log("\U0001f4a1", "Make sure you're running this on Databricks compute.")
    raise SystemExit(1)

_api_base = w.config.host.rstrip("/") + "/api/2.0"

def _get_api_headers():
    """Build auth headers using the SDK's authenticate method (works on all compute types)."""
    h = {"Content-Type": "application/json"}
    h.update(w.config.authenticate())
    return h

_api_headers = _get_api_headers()

# ---------------------------------------------------------------------------
# STEP 1: Discover SQL Warehouse
# ---------------------------------------------------------------------------
log("\U0001f50d", "Discovering SQL Warehouses...")
warehouse_id = _override_warehouse
warehouse_name = "(override)"

if not warehouse_id:
    try:
        warehouses = list(w.warehouses.list())
        if not warehouses:
            log("\u274c", "No SQL Warehouses found in this workspace.")
            log("\U0001f4a1", "Please create a SQL Warehouse and rerun this cell.")
            raise SystemExit(1)

        # Priority 1: Look for Serverless Starter Warehouse (available by default)
        best = None
        for wh in warehouses:
            wh_name = getattr(wh, 'name', '') or ''
            if 'starter' in wh_name.lower() and 'serverless' in wh_name.lower():
                best = wh
                break

        # Priority 2: Fall back to best available (RUNNING > STARTING > STOPPED, SERVERLESS > PRO > CLASSIC)
        if not best:
            type_priority = {"SERVERLESS": 0, "PRO": 1, "CLASSIC": 2}
            state_priority = {"RUNNING": 0, "STARTING": 1, "STOPPED": 2, "STOPPING": 3, "DELETED": 99, "DELETING": 99}

            def wh_sort_key(wh):
                s = state_priority.get(str(getattr(wh, 'state', 'STOPPED')), 5)
                t = type_priority.get(str(getattr(wh, 'warehouse_type', 'CLASSIC')), 2)
                return (s, t)

            warehouses.sort(key=wh_sort_key)
            best = warehouses[0]

        warehouse_id = best.id
        warehouse_name = best.name
        wh_state = str(getattr(best, 'state', 'UNKNOWN'))
        wh_type = str(getattr(best, 'warehouse_type', 'UNKNOWN'))
        log("\u2705", f'Found SQL Warehouse: "{warehouse_name}" (id: {warehouse_id}, {wh_type}, {wh_state})')
    except SystemExit:
        raise
    except Exception as e:
        log("\u274c", f"Error discovering warehouses: {e}")
        raise
else:
    log("\u2705", f"Using override warehouse: {warehouse_id}")

# ---------------------------------------------------------------------------
# STEP 2: Discover Claude Serving Endpoints (Dynamic — no hardcoded names)
# ---------------------------------------------------------------------------
log("\U0001f50d", "Discovering Claude LLM Serving Endpoints...")
llm_endpoints = _override_endpoints

if not llm_endpoints:
    try:
        all_eps = list(w.serving_endpoints.list())

        # Match ANY endpoint with 'claude' in its name — fully dynamic
        candidates = [
            ep.name for ep in all_eps
            if ep.name and re.search(r"claude", ep.name, re.IGNORECASE)
        ]

        def ep_sort_key(name):
            n = name.lower()
            # Tier priority: opus=0, sonnet=1, haiku=2, other=3
            if "opus" in n: tier = 0
            elif "sonnet" in n: tier = 1
            elif "haiku" in n: tier = 2
            else: tier = 3
            # Extract version numbers after 'claude', negate for descending (latest first)
            nums = re.findall(r"(\d+)", n.split("claude")[-1])
            version = tuple(-int(x) for x in nums) if nums else (0,)
            return (tier, version, name)

        candidates.sort(key=ep_sort_key)
        llm_endpoints = candidates  # Use ALL discovered Claude endpoints

        if not llm_endpoints:
            log("\u274c", "No Claude endpoints found in this workspace.")
            log("\U0001f4a1", "Enable Foundation Model APIs (Claude) in your workspace and rerun.")
            log("\U0001f4a1", "Or set LLM_ENDPOINTS in Cell 2 with your custom endpoint names.")
            raise SystemExit(1)

        log("\u2705", f"Found {len(llm_endpoints)} Claude endpoint(s)")
    except SystemExit:
        raise
    except Exception as e:
        log("\u274c", f"Error discovering endpoints: {e}")
        raise
else:
    log("\u2705", f"Using override endpoints: {', '.join(llm_endpoints)}")

# ---------------------------------------------------------------------------
# STEP 3: Detect Source Path (project root + app subfolder)
# ---------------------------------------------------------------------------
log("\U0001f4c1", "Detecting source code path...")
try:
    nb_path = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().notebookPath().get()
    )
    project_root = "/Workspace" + "/".join(nb_path.split("/")[:-1])
    source_path = project_root + "/" + APP_SUBFOLDER
    log("\u2705", f"Project root: {project_root}")
    log("\u2705", f"App source:   {source_path}")
except Exception:
    try:
        project_root = os.path.dirname(os.path.abspath("__file__"))
        if not project_root.startswith("/Workspace"):
            project_root = "/Workspace" + project_root
        source_path = project_root + "/" + APP_SUBFOLDER
        log("\u2705", f"App source (fallback): {source_path}")
    except Exception as e:
        log("\u274c", f"Cannot detect source path: {e}")
        log("\U0001f4a1", "Make sure this notebook is inside the project directory.")
        raise

# ---------------------------------------------------------------------------
# STEP 4: Create or Update Databricks App (REST API)
# ---------------------------------------------------------------------------
log("\U0001f528", f'Creating Databricks App: "{APP_NAME}"...')
app_exists = False
_check = requests.get(f"{_api_base}/apps/{APP_NAME}", headers=_api_headers, timeout=30)
if _check.status_code == 200:
    app_exists = True
    log("\u2139\ufe0f", "App already exists -- will update and redeploy.")
else:
    try:
        resp = requests.post(
            f"{_api_base}/apps",
            headers=_api_headers,
            json={"name": APP_NAME, "description": APP_TITLE + " -- AI-powered dimensional model designer for Unity Catalog schemas"},
            timeout=120,
        )
        if resp.status_code == 200:
            log("\u2705", "App created successfully")
        elif resp.status_code == 409 or "already exists" in resp.text.lower():
            app_exists = True
            log("\u2139\ufe0f", "App already exists -- will update and redeploy.")
        else:
            body = resp.text
            if "maximum limit" in body.lower():
                log("\u26a0\ufe0f", "Workspace at 200 app limit. Searching for unused apps to auto-cleanup...")
                _freed = False
                try:
                    _apps_resp = requests.get(f"{_api_base}/apps", headers=_api_headers, timeout=60)
                    if _apps_resp.status_code == 200:
                        _all_apps = _apps_resp.json().get("apps", [])
                        _del_candidates = []
                        for _a in _all_apps:
                            _aname = _a.get("name", "")
                            _active = _a.get("active_deployment")
                            _st = (_a.get("status") or {}).get("state", "").upper()
                            if _st in ("ERROR", "FAILED", "UNAVAILABLE", "DELETED"):
                                _del_candidates.insert(0, _aname)
                            elif not _active and _aname != APP_NAME:
                                _del_candidates.append(_aname)
                        if _del_candidates:
                            _to_del = _del_candidates[0]
                            log("\U0001f5d1\ufe0f", f'Deleting unused app "{_to_del}" to free a slot...')
                            _dr = requests.delete(f"{_api_base}/apps/{_to_del}", headers=_api_headers, timeout=60)
                            if _dr.status_code in (200, 204):
                                log("\u2705", f'Deleted "{_to_del}". Retrying app creation...')
                                time.sleep(3)
                                resp = requests.post(
                                    f"{_api_base}/apps",
                                    headers=_api_headers,
                                    json={"name": APP_NAME, "description": APP_TITLE + " -- AI-powered dimensional model designer"},
                                    timeout=120,
                                )
                                if resp.status_code == 200:
                                    log("\u2705", "App created successfully (after cleanup)")
                                    _freed = True
                                else:
                                    log("\u274c", f"Retry still failed: {resp.text[:200]}")
                            else:
                                log("\u274c", f'Could not delete "{_to_del}": {_dr.status_code}')
                        else:
                            log("\u274c", "No unused apps found to auto-delete.")
                except Exception as _ce:
                    log("\u274c", f"Auto-cleanup error: {_ce}")
                if not _freed:
                    log("\u274c", "Cannot create app -- workspace at 200 app limit.")
                    log("\U0001f4a1", "Manually delete unused apps from the Apps page and rerun.")
                    raise SystemExit(1)
            else:
                resp.raise_for_status()
    except (requests.exceptions.HTTPError, SystemExit) as e:
        if isinstance(e, SystemExit):
            raise
        log("\u274c", f"Failed to create app: {e}")
        if "permission" in str(e).lower() or "403" in str(e):
            log("\U0001f4a1", "You may need workspace admin privileges to create apps.")
        raise

# ---------------------------------------------------------------------------
# STEP 4b: Wait for App Compute to be Ready
# ---------------------------------------------------------------------------
log("\u23f3", "Waiting for app compute to be ready (this can take 1-3 minutes)...")
_compute_ready = False
for _wait_i in range(90):  # Up to ~7.5 minutes
    try:
        _wr = requests.get(f"{_api_base}/apps/{APP_NAME}", headers=_api_headers, timeout=30)
        if _wr.status_code == 200:
            _winfo = _wr.json()
            _cstate = _winfo.get("compute_status", {}).get("state", "UNKNOWN")
            if _cstate in ("ACTIVE", "RUNNING"):
                log("\u2705", f"App compute is ready (state: {_cstate})")
                _compute_ready = True
                break
            elif _cstate in ("ERROR", "FAILED", "TERMINATED"):
                _cmsg = _winfo.get("compute_status", {}).get("message", "")
                log("\u274c", f"App compute failed: {_cstate} -- {_cmsg}")
                log("\U0001f4a1", "Try deleting the app and rerunning the installer.")
                raise SystemExit(1)
            else:
                if _wait_i % 6 == 0:  # Log every 30 seconds
                    elapsed = _wait_i * 5
                    log("\u23f3", f"Compute state: {_cstate} ({elapsed}s elapsed)")
    except SystemExit:
        raise
    except Exception:
        pass
    time.sleep(5)

if not _compute_ready:
    log("\u26a0\ufe0f", "Compute readiness check timed out. Proceeding anyway...")

# ---------------------------------------------------------------------------
# STEP 5: Configure Resources & Permissions (REST API + SDK fallback)
# ---------------------------------------------------------------------------
log("\U0001f517", "Configuring app resources and permissions...")
primary_ep = llm_endpoints[0]  # Best Claude endpoint (dynamically discovered)
_resources_ok = False

# --- Attempt 1: REST API PATCH ---
try:
    update_payload = {
        "name": APP_NAME,
        "description": APP_TITLE + " -- AI-powered dimensional model designer for Unity Catalog schemas",
        "resources": [
            {"name": "sql-warehouse", "sql_warehouse": {"id": warehouse_id, "permission": "CAN_USE"}},
            {"name": "serving-endpoint", "serving_endpoint": {"name": primary_ep, "permission": "CAN_QUERY"}},
        ],
        "user_api_scopes": [
            "sql", "serving.serving-endpoints",
            "catalog.catalogs:read", "catalog.schemas:read", "catalog.tables:read",
        ],
    }
    resp = requests.patch(
        f"{_api_base}/apps/{APP_NAME}",
        headers=_api_headers,
        json=update_payload,
        timeout=60,
    )
    resp.raise_for_status()
    _resources_ok = True
    log("\u2705", f"SQL Warehouse attached: {warehouse_name} (CAN_USE)")
    log("\u2705", f"Serving Endpoint attached: {primary_ep} (CAN_QUERY)")
    log("\u2705", "API scopes configured: sql, serving.serving-endpoints, catalog.*")
except Exception as e1:
    log("\u26a0\ufe0f", f"REST API config failed: {e1}")

    # --- Attempt 2: Databricks SDK fallback ---
    try:
        from databricks.sdk.service.apps import (
            App, AppResource, AppResourceSqlWarehouse,
            AppResourceSqlWarehouseSqlWarehousePermission,
            AppResourceServingEndpoint,
            AppResourceServingEndpointServingEndpointPermission,
        )
        log("\U0001f504", "Retrying with Databricks SDK...")
        w.apps.update(
            name=APP_NAME,
            app=App(
                name=APP_NAME,
                description=APP_TITLE + " -- AI-powered dimensional model designer for Unity Catalog schemas",
                resources=[
                    AppResource(name="sql-warehouse", sql_warehouse=AppResourceSqlWarehouse(
                        id=warehouse_id,
                        permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE)),
                    AppResource(name="serving-endpoint", serving_endpoint=AppResourceServingEndpoint(
                        name=primary_ep,
                        permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY)),
                ],
                user_api_scopes=["sql", "serving.serving-endpoints",
                                 "catalog.catalogs:read", "catalog.schemas:read", "catalog.tables:read"],
            )
        )
        _resources_ok = True
        log("\u2705", "Resources configured via SDK")
    except Exception as e2:
        log("\u274c", f"SDK config also failed: {e2}")

if not _resources_ok:
    # Print detailed manual setup instructions
    _apps_ui_url = w.config.host.rstrip('/') + f"/apps/{APP_NAME}"
    log("\u26a0\ufe0f", "")
    print("\n" + "\u2501" * 50, flush=True)
    print("  \U0001f6e0  MANUAL SETUP REQUIRED", flush=True)
    print("\u2501" * 50, flush=True)
    print(f"""
  The app was created and deployed, but resources could
  not be attached automatically (likely a permissions issue).

  Without these resources, the app CANNOT access catalogs,
  run SQL, or call LLM endpoints.

  \u2794 Open the app settings page and configure manually:
    {_apps_ui_url}

  STEP 1: Add Resources (Settings > Resources)
  \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    Resource 1:
      Key:        sql-warehouse
      Type:       SQL Warehouse
      Warehouse:  {warehouse_name} (ID: {warehouse_id})
      Permission: CAN_USE

    Resource 2:
      Key:        serving-endpoint
      Type:       Serving Endpoint
      Endpoint:   {primary_ep}
      Permission: CAN_QUERY

  STEP 2: Add API Scopes (Settings > Permissions)
  \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    Add these User API Scopes:
      - sql
      - serving.serving-endpoints
      - catalog.catalogs:read
      - catalog.schemas:read
      - catalog.tables:read

  STEP 3: Grant Catalog Access to App Service Principal
  \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    In Catalog Explorer, grant the app's service principal
    USE CATALOG + USE SCHEMA + SELECT on the catalogs
    you want to analyze. The service principal name is
    shown on the app's settings page.

  After completing these steps, refresh the app page.
""", flush=True)
    print("\u2501" * 50 + "\n", flush=True)

# ---------------------------------------------------------------------------
# STEP 6: Deploy (REST API)
# ---------------------------------------------------------------------------
log("\U0001f6a2", f"Deploying app from {source_path} (this may take 2-5 minutes)...")
try:
    resp = requests.post(
        f"{_api_base}/apps/{APP_NAME}/deployments",
        headers=_api_headers,
        json={"source_code_path": source_path},
        timeout=120,
    )
    resp.raise_for_status()

    deploy_start = time.time()
    last_status = ""
    while True:
        elapsed = time.time() - deploy_start
        try:
            r = requests.get(f"{_api_base}/apps/{APP_NAME}", headers=_api_headers, timeout=30)
            if r.status_code == 200:
                info = r.json()
                # Check both pending and active deployments
                current = info.get("pending_deployment") or info.get("active_deployment") or {}
                status_obj = current.get("status", {})
                status = status_obj.get("state", "PENDING")
                app_status = info.get("app_status", {}).get("state", "")
            else:
                status = "PENDING"
                app_status = ""

            if status != last_status or app_status == "RUNNING":
                if status in ("SUCCEEDED",) or app_status == "RUNNING":
                    break
                elif status in ("FAILED", "CANCELLED"):
                    msg = status_obj.get("message", "Unknown error")
                    log("\u274c", f"Deployment failed: {status} -- {msg}")
                    raise SystemExit(1)
                else:
                    mins = int(elapsed // 60)
                    secs = int(elapsed % 60)
                    log("\u23f3", f"Deployment status: {status} ({mins}m {secs:02d}s elapsed)")
                    last_status = status
            else:
                if int(elapsed) % 30 < 10 and elapsed > 15:
                    mins = int(elapsed // 60)
                    secs = int(elapsed % 60)
                    log("\u23f3", f"Still deploying... ({mins}m {secs:02d}s elapsed)")

        except SystemExit:
            raise
        except Exception:
            pass

        if elapsed > 600:
            log("\u26a0\ufe0f", "Deployment is taking longer than expected (>10 min).")
            log("\U0001f4a1", f"Check the Apps page manually for app '{APP_NAME}'.")
            break

        time.sleep(10)

    total_elapsed = time.time() - deploy_start
    mins = int(total_elapsed // 60)
    secs = int(total_elapsed % 60)
    log("\u2705", f"Deployment complete! ({mins}m {secs:02d}s)")

except SystemExit:
    raise
except Exception as e:
    log("\u274c", f"Deployment error: {e}")
    log("\U0001f4a1", "Check the Apps page for more details.")

# ---------------------------------------------------------------------------
# STEP 7: Print App URL
# ---------------------------------------------------------------------------
try:
    r = requests.get(f"{_api_base}/apps/{APP_NAME}", headers=_api_headers, timeout=30)
    app_url = r.json().get("url", "") if r.status_code == 200 else ""
    if not app_url:
        app_url = w.config.host.rstrip("/") + f"/apps/{APP_NAME}"
except Exception:
    app_url = w.config.host.rstrip("/") + f"/apps/{APP_NAME}"

total_time = time.time() - start_time
mins = int(total_time // 60)
secs = int(total_time % 60)

if _resources_ok:
    banner(f"\U0001f389 APP IS READY!  (Total: {mins}m {secs:02d}s)")
else:
    banner(f"\U0001f389 APP DEPLOYED!  (Total: {mins}m {secs:02d}s) \u2014 manual config needed")
print(f"  \U0001f517 {app_url}\n", flush=True)
if not _resources_ok:
    print(f"  \u26a0 Complete the manual setup steps above before using the app.", flush=True)
else:
    print(f"  Open the URL above to start designing data models.", flush=True)
print("\n" + "\u2501" * 50 + "\n", flush=True)

try:
    if _resources_ok:
        displayHTML(f'''
        <div style="padding:20px; background:#0f172a; border-radius:12px; text-align:center; margin:10px 0;">
            <h2 style="color:#86efac; margin:0 0 10px 0;">\U0001f389 App Deployed Successfully!</h2>
            <a href="{app_url}" target="_blank"
               style="color:#60a5fa; font-size:20px; text-decoration:none; font-weight:700;">
               \U0001f517 {APP_TITLE}
            </a>
            <p style="color:#64748b; margin:8px 0 0 0; font-size:12px;">{app_url}</p>
        </div>
        ''')
    else:
        _apps_settings = w.config.host.rstrip('/') + f"/apps/{APP_NAME}"
        displayHTML(f'''
        <div style="padding:20px; background:#0f172a; border-radius:12px; text-align:center; margin:10px 0;">
            <h2 style="color:#fbbf24; margin:0 0 10px 0;">\u26a0 App Deployed \u2014 Manual Config Needed</h2>
            <p style="color:#94a3b8; margin:0 0 12px 0; font-size:13px;">Resources could not be attached automatically. Configure them in the Apps UI to enable catalog access.</p>
            <a href="{_apps_settings}" target="_blank"
               style="display:inline-block; padding:10px 24px; background:#2563eb; color:white; border-radius:8px;
                      font-size:14px; text-decoration:none; font-weight:600; margin:0 8px;">
               \U0001f6e0 Open App Settings
            </a>
            <a href="{app_url}" target="_blank"
               style="display:inline-block; padding:10px 24px; background:#1e293b; color:#60a5fa; border-radius:8px;
                      font-size:14px; text-decoration:none; font-weight:600; margin:0 8px; border:1px solid #334155;">
               \U0001f517 {APP_TITLE}
            </a>
        </div>
        ''')
except Exception:
    pass